# 1、content的使用

举例1：保存多模态的数据，使用字典列表

In [1]:
import base64
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage
from dotenv import load_dotenv
import os

load_dotenv(override=True)

model = init_chat_model(
    model="gpt-5.4-mini",
    model_provider="openai",
    api_key=os.getenv("CLOSEAI_API_KEY"),
    base_url=os.getenv("CLOSEAI_BASE_URL")
)

def encode_image(img_path, img_type='jpeg'):
    """将一张本地图片转换成 Base64 编码的 Data URI 字符串,方便在文本中嵌入图片数据"""
    with open(img_path, "rb") as img_file:
        return f"data:image/{img_type};base64,{base64.b64encode(img_file.read()).decode("utf-8")}"

# 图像路径
img_path = "image_test.png"

# 获取图像base64编码字符串
base64_image = encode_image(img_path)

response = model.invoke(
    [
        HumanMessage(
            content=[
                {'type': 'text', 'text': '这张图里有什么？'},
                {
                    'type': 'image_url',
                    "image_url": base64_image,
                }
            ]
        )
    ]
)
print(response.content)

图中是一瓶香水，摆放在浅米色/金色背景上，整体光线很柔和，带有投影效果。

它看起来像：
- 透明方形玻璃瓶身
- 金色瓶盖和装饰
- 瓶内是浅金色液体
- 风格偏高档、奢华、精致

如果你愿意，我也可以帮你进一步描述它的品牌风格、拍摄构图，或者提取适合商品文案的描述。


# 2、content_blocks的使用

举例1 ：输入的格式化

In [4]:
from langchain.messages import HumanMessage
import os
from dotenv import load_dotenv

load_dotenv(override=True)

model = init_chat_model(
    model="gpt-5.4-mini",
    model_provider="openai",
    api_key=os.getenv("CLOSEAI_API_KEY"),
    base_url=os.getenv("CLOSEAI_BASE_URL")
)

def encode_image(img_path):
    """将一张本地图片转换成 Base64 编码的 Data URI 字符串,方便在文本中嵌入图片数据"""
    with open(img_path, "rb") as img_file:
        return base64.b64encode(img_file.read()).decode("utf-8")

# 图像路径
img_path = "image_test.png"

# 获取图像base64编码字符串
base64_image = encode_image(img_path)

response = model.invoke(
    [
        # 此种格式可用
        # HumanMessage(
        #     content=[
        #         {'type': 'text', 'text': '这张图里有什么？'},
        #         {
        #             'type': 'image_url',
        #             "image_url": base64_image,
        #         }
        #     ]
        # )
        # 推荐的统一写法
        HumanMessage(
            content_blocks=[
                {'type': 'text', 'text': '这张图里有什么？'},
                {
                    'type': 'image',
                    'base64': base64_image,
                    'mime_type': 'image/png',
                }
            ]
        )

    ]
)
print(response.content)

图里是一瓶香水。  
它是透明方形玻璃瓶身，金色瓶盖和装饰，摆在浅米色背景上，整体光线很柔和，偏高级感、简洁风。


作为对比

In [3]:
import base64
from langchain.messages import HumanMessage

from dotenv import load_dotenv
load_dotenv(override=True)

model = init_chat_model(
    model="claude-haiku-4-5",
    model_provider="openai",
    api_key=os.getenv("CLOSEAI_API_KEY"),
    base_url=os.getenv("CLOSEAI_BASE_URL")
)

def encode_image(img_path):
    """将一张本地图片转换成 Base64 编码的 Data URI 字符串,方便在文本中嵌入图片数据"""
    with open(img_path, "rb") as img_file:
        return base64.b64encode(img_file.read()).decode("utf-8")

# 图像路径
img_path = "image_test.png"

# 获取图像base64编码字符串
base64_image = encode_image(img_path)

response = model.invoke(
    [
        # 传统的写法：使用content。读取失败
        # HumanMessage(
        #     content=[
        #         {'type': 'text', 'text': '这张图里有什么？'},
        #         {
        #             'type': 'image_url',
        #             "image_url": base64_image,
        #         }
        #     ]
        # )

        # 推荐的统一写法
        HumanMessage(
            content_blocks=[
                {'type': 'text', 'text': '这张图里有什么？'},
                {
                    'type': 'image',
                    'base64': base64_image,
                    'mime_type': 'image/png',
                }
            ]
        )
    ]
)
print(response.content)

# 图片内容描述

这张图里有一个**化妆品瓶子**，具体来说是：

- **类型**：液体粉底液或化妆水产品
- **颜色**：浅米色/肤色液体
- **瓶身**：透明玻璃瓶，可以看到内部产品
- **瓶盖**：金色的方形泵头设计
- **摆放方式**：竖直放置在浅米色/杏色的纯色背景上
- **光影效果**：有明显的阴影，展示出产品的立体感和质感

这是典型的**美妆产品宣传照**，拍摄角度简洁专业，突出了产品的质感和高端感。


举例2：输出的格式化

In [7]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from rich import print as rprint
load_dotenv()

load_dotenv(override=True)

model = init_chat_model(
    model="deepseek:deepseek-v4-flash",
    extra_body={"thinking": {"type": "enabled"}},
)

response = model.invoke("你好，一句话回答")
rprint('=' * 20, '-> response <-', '=' * 20)
rprint(response)
rprint('=' * 20, '-> response.content <-', '=' * 20)
rprint(response.content)
rprint('=' * 20, '-> response.content_blocks <-', '=' * 20)
rprint(response.content_blocks)

==================== -> response <- ====================

AIMessage(
    content='你好！有什么可以帮你的吗？',
    additional_kwargs={
        'refusal': None,
        'reasoning_content': 
'我们需要回答用户“你好，一句话回答”。需要一句话回答。用中文简短问候。注意只一句话。可以：“你好！有什么可以帮你的吗
？” 这是一句话包含问号。确保一句话。最终只输出一句。'
    },
    response_metadata={
        'token_usage': {
            'completion_tokens': 56,
            'prompt_tokens': 34,
            'total_tokens': 90,
            'completion_tokens_details': {
                'accepted_prediction_tokens': None,
                'audio_tokens': None,
                'reasoning_tokens': 47,
                'rejected_prediction_tokens': None
            },
            'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0},
            'prompt_cache_hit_tokens': 0,
            'prompt_cache_miss_tokens': 34
        },
        'model_provider': 'deepseek',
        'model_name': 'deepseek-flash',
        'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6',
        'id': 'f6417bc3-8ddb-40ca-baca-9278cab61564',
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='lc_run--01a19b5f-aa97-760c-90c8-a4dba8861d2f-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 34,
        'output_tokens': 56,
        'total_tokens': 90,
        'input_token_details': {'cache_read': 0},
        'output_token_details': {'reasoning': 47}
    }
)

==================== -> response.content <- ====================

你好！有什么可以帮你的吗？

==================== -> response.content_blocks <- ====================

[
    {
        'type': 'reasoning',
        'reasoning': 
'我们需要回答用户“你好，一句话回答”。需要一句话回答。用中文简短问候。注意只一句话。可以：“你好！有什么可以帮你的吗
？” 这是一句话包含问号。确保一句话。最终只输出一句。'
    },
    {'type': 'text', 'text': '你好！有什么可以帮你的吗？'}
]